In [4]:
from google.colab import files
uploaded = files.upload()

Saving archive (5).zip to archive (5).zip


In [6]:
import zipfile

zip_path = "archive (5).zip"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall("data_folder")

In [7]:
import os

print(os.listdir("data_folder"))

['data', 'Resume']


In [9]:
print(os.listdir("data_folder/Resume"))

['Resume.csv']


In [11]:
import pandas as pd

df = pd.read_csv("data_folder/Resume/Resume.csv")
print(df.head())

         ID                                         Resume_str  \
0  16852973           HR ADMINISTRATOR/MARKETING ASSOCIATE\...   
1  22323967           HR SPECIALIST, US HR OPERATIONS      ...   
2  33176873           HR DIRECTOR       Summary      Over 2...   
3  27018550           HR SPECIALIST       Summary    Dedica...   
4  17812897           HR MANAGER         Skill Highlights  ...   

                                         Resume_html Category  
0  <div class="fontsize fontface vmargins hmargin...       HR  
1  <div class="fontsize fontface vmargins hmargin...       HR  
2  <div class="fontsize fontface vmargins hmargin...       HR  
3  <div class="fontsize fontface vmargins hmargin...       HR  
4  <div class="fontsize fontface vmargins hmargin...       HR  


In [12]:
print(df.columns)

Index(['ID', 'Resume_str', 'Resume_html', 'Category'], dtype='object')


In [13]:
resumes = df['Resume_str']

In [14]:
import re
import nltk
from nltk.corpus import stopwords

nltk.download('stopwords')

def clean_text(text):
    text = text.lower()
    text = re.sub(r'\W', ' ', text)
    text = re.sub(r'\d', '', text)
    words = text.split()
    words = [w for w in words if w not in stopwords.words('english')]
    return " ".join(words)

resumes_clean = resumes.apply(clean_text)

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [15]:
job_description = """
Looking for a Data Scientist with skills in Python, Machine Learning, SQL, Data Analysis
"""

jd_clean = clean_text(job_description)

In [16]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

scores = []

for resume in resumes_clean:
    tfidf = TfidfVectorizer()
    matrix = tfidf.fit_transform([jd_clean, resume])

    score = cosine_similarity(matrix[0:1], matrix[1:2])[0][0]
    scores.append(score)

In [17]:
df['score'] = scores

df_sorted = df.sort_values(by='score', ascending=False)

print(df_sorted[['Category', 'score']].head())

                    Category     score
1218              CONSULTANT  0.351492
1762             ENGINEERING  0.287454
331   INFORMATION-TECHNOLOGY  0.276150
1339              AUTOMOBILE  0.261505
1091                   SALES  0.257455


In [18]:
skills = ["python", "machine learning", "sql", "data analysis"]

def extract_skills(text):
    return [skill for skill in skills if skill in text]

df['skills'] = resumes_clean.apply(extract_skills)

In [19]:
jd_skills = skills

def skill_gap(resume_skills):
    return list(set(jd_skills) - set(resume_skills))

df['missing_skills'] = df['skills'].apply(skill_gap)

In [21]:
df_sorted = df.sort_values(by='score', ascending=False)
print(df_sorted[['Category', 'score', 'skills', 'missing_skills']].head())

                    Category     score  \
1218              CONSULTANT  0.351492   
1762             ENGINEERING  0.287454   
331   INFORMATION-TECHNOLOGY  0.276150   
1339              AUTOMOBILE  0.261505   
1091                   SALES  0.257455   

                                              skills  \
1218  [python, machine learning, sql, data analysis]   
1762  [python, machine learning, sql, data analysis]   
331                     [python, sql, data analysis]   
1339                    [python, sql, data analysis]   
1091                                           [sql]   

                                 missing_skills  
1218                                         []  
1762                                         []  
331                          [machine learning]  
1339                         [machine learning]  
1091  [machine learning, data analysis, python]  
